In [ ]:
from datasets import load_dataset
import json

### processing toxic data

In [ ]:
dataset = load_dataset('json', data_files='/data/chaojian/Multi-alignment/dataset/toxic_data.jsonl')

In [ ]:
def fix_data(example):
    return {

        k: v.replace("\\n", "\n") if isinstance(v, str) else v
        for k, v in example.items()
    }

dataset = dataset.map(fix_data)

In [ ]:
dataset.save_to_disk('/data/chaojian/Multi-alignment/dataset/toxic_alignment')

### processing stereotype data

In [ ]:
dataset = load_dataset('json', data_files='/data/chaojian/Multi-alignment/dataset/merged_stereotype.jsonl')



In [ ]:
dataset = dataset.map(fix_data)

In [ ]:
dataset['train'][0]

In [ ]:
dataset.save_to_disk('/data/chaojian/Multi-alignment/dataset/sterotype_alignment')

### processing safety data

In [ ]:
dataset = load_dataset('json', data_files='/data/chaojian/Multi-alignment/dataset/merged_pku_safety.jsonl')

In [ ]:
dataset = dataset.map(fix_data)

dataset['train'][0]

In [ ]:
dataset.save_to_disk('/data/chaojian/Multi-alignment/dataset/pku_safety_alignment')

### processing moral(ethic) data   

In [ ]:
from datasets import load_from_disk

In [ ]:
dataset = load_from_disk('/data/chaojian/Multi-alignment/dataset/moral_stories')
dataset

In [ ]:
dataset['train'][0]['situation']

In [ ]:
dataset['train'][0]['intention']

In [ ]:
if dataset['train'][1]['moral_action']:
    print(True)
else:
    print(False)

In [ ]:
dataset_filter = dataset.filter(lambda example: example['moral_action'] != None)
dataset_filter

In [ ]:
(dataset_filter['train'][0]['situation'], dataset_filter['train'][0]['intention'], dataset_filter['train'][0]['moral_action'])

In [ ]:
dataset_filter['train'][0]['norm'], dataset_filter['train'][0]['moral_consequence']

In [ ]:
import random
from datasets import DatasetDict

def build_full_output(norm, action, consequence):
    norm = norm.strip()
    action = action.strip()
    consequence = consequence.strip()
    
    norm_intro = random.choice([
        f"Because {norm},",
        f"Since {norm},",
        f"As people believe that {norm},",
        f"In line with the idea that {norm},",
        f"Given that {norm},",
        f"Considering that {norm},",
    ])
    
    consequence_intro = random.choice([
        "As a result,",
        "This leads to the fact that",
        "Consequently,",
        "Therefore,",
        "This results in",
    ])
    
    return f"{norm_intro} {action} {consequence_intro} {consequence}"

def convert_example_with_consequence(example):
    input_text = (
        f"Situation: {example['situation'].strip()} "
        f"Intention: {example['intention'].strip()} "
        f"Question: What should he or she do?"
    )
    output_text = build_full_output(
        example['norm'][:-1],     # 去除原有的逗号
        example['moral_action'],
        example['moral_consequence']
    )
    
    return {
        "input": input_text,
        "full_output": output_text
    }

sft_dataset = dataset_filter.map(convert_example_with_consequence, remove_columns=dataset_filter['train'].column_names)

In [ ]:
sft_dataset['train'][0], sft_dataset['validation'][0], sft_dataset['test'][0] 


In [ ]:
sft_dataset.save_to_disk('/data/chaojian/Multi-alignment/dataset/alignment_moral')

### processing truth data

In [ ]:
from datasets import load_dataset

ds = load_dataset("zwhe99/commonsense_170k")

In [ ]:
ds['train'][0]

In [ ]:
def process_commonsense(examples):

    return{
        'input':examples['instruction'],
        'full_output':examples['answer']
    }

sft_data = ds.map(process_commonsense, remove_columns=ds['train'].column_names)

In [ ]:
sft_data['train'][0]

In [ ]:
sft_data.save_to_disk('/data/chaojian/Multi-alignment/dataset/alignment_truthful')

### processing helpful/instruction following data

In [ ]:
ds = load_dataset('openbmb/UltraFeedback')

In [ ]:
ds_sharegpt = ds['train'].filter(lambda x: x['source'] == 'sharegpt')

In [ ]:
ds_ultrachat = ds['train'].filter(lambda x: x['source'] == 'ultrachat')

In [ ]:
ds_ultrachat['instruction'][0:5]

In [ ]:
ds_ultrachat

In [ ]:
def get_most_helpful(example, min_instruction_following=4):
    completions = example["completions"]
    best_completion = None
    best_helpfulness_score = -1

    for c in completions:
        try:
            helpfulness_score = int(c["annotations"]["helpfulness"]["Rating"])
            instruction_following_score = int(
                c["annotations"]["instruction_following"]["Rating"]
            )
        except:
            continue

        if instruction_following_score >= min_instruction_following:
            if helpfulness_score > best_helpfulness_score:
                best_helpfulness_score = helpfulness_score
                best_completion = c

    if best_completion:
        return {
            "input": example["instruction"],
            "full_output": best_completion["response"],
            "helpfulness_score": best_helpfulness_score,
            "instruction_following_score": int(
                best_completion["annotations"]["instruction_following"]["Rating"]
            )
        }
    else:
        return {
            "input": example["instruction"],
            "full_output": None,
            "helpfulness_score": None,
            "instruction_following_score": None
        }
    
most_helpful_ultrachat = ds_ultrachat.map(
    lambda x: get_most_helpful(x, min_instruction_following=4),
    remove_columns=ds_ultrachat.column_names
)
most_helpful_ultrachat = most_helpful_ultrachat.filter(lambda x: x["full_output"] is not None)


In [ ]:
most_helpful_ultrachat['helpfulness_score'][1], most_helpful_ultrachat['instruction_following_score'][1]

In [ ]:
most_helpful_ultrachat

In [ ]:
most_helpful_sharegpt = ds_sharegpt.map(
    lambda x: get_most_helpful(x, min_instruction_following=4),
    remove_columns=ds_sharegpt.column_names
)
most_helpful_sharegpt = most_helpful_sharegpt.filter(lambda x: x["full_output"] is not None)

In [ ]:
most_helpful_sharegpt['helpfulness_score'][1], most_helpful_sharegpt['instruction_following_score'][1]

In [ ]:
ds_evol = ds['train'].filter(lambda x: x['source'] == 'evol_instruct')

In [ ]:
most_helpful_evol = ds_evol.map(
    lambda x: get_most_helpful(x, min_instruction_following=4),
    remove_columns=ds_evol.column_names
)

most_helpful_evol = most_helpful_evol.filter(lambda x: x["full_output"] is not None)

In [ ]:
from datasets import concatenate_datasets
ultra_helpful_concat = concatenate_datasets([most_helpful_ultrachat, most_helpful_evol, most_helpful_sharegpt])

In [ ]:
ultra_helpful_concat.save_to_disk("/data/chaojian/Multi-alignment/dataset/alignment_helpfulness")

In [ ]:
ds.save_to_disk('/data/chaojian/Multi-alignment/dataset/ultrafeedback')